# Laboratorio 05 - Soluciones y Ejemplos Completos

**Soluciones detalladas para: Construir un dashboard completo en Power BI**

---

## Objetivo de este documento

Este notebook proporciona:
1. Medidas DAX completas y probadas
2. Ejemplos de configuraciones en Python (para verificar cálculos)
3. Screenshots conceptuales de lo esperado
4. Errores comunes y cómo evitarlos
5. Extensiones opcionales para usuarios avanzados

---

## Solución Paso 1: Transformaciones en Power Query

### Transformaciones esperadas

| Columna | Tipo Correcto | Acción requerida |
|---------|--------------|------------------|
| `id_estudiante` | Texto o Entero | Dejar como está |
| `carrera` | Texto | Sin cambios |
| `semestre` | Número entero | Verificar que no sea texto |
| `nota` | Número decimal | Cambiar si importó como texto |
| `estado` | Texto | Sin cambios |
| `periodo` | Texto o Fecha | Si contiene fechas, convertir |
| `trabaja` | Texto (Sí/No) | Podría convertir a booleano |

### Errores comunes en Power Query

1. **Columnas numéricas como texto**: Ocurre cuando el CSV tiene espacios o caracteres especiales
   - **Solución**: Reemplazar errores → Cambiar tipo

2. **Delimitador incorrecto**: El archivo usa `;` pero Power BI asume `,`
   - **Solución**: En la pantalla de importación, cambiar el delimitador

3. **Primera fila como datos**: Los encabezados se mezclan con datos
   - **Solución**: Usar como encabezados

In [ ]:
# Verificar la estructura del dataset para comparar con Power BI
import pandas as pd
import numpy as np

# Cargar datos
df = pd.read_csv('../datasets/universidad/rendimiento_academico.csv')

print("VERIFICACIÓN DE TIPOS - Compare con Power BI")
print("="*60)
print()

for col in df.columns:
    tipo_python = df[col].dtype
    tipo_pbi = ""
    if tipo_python == 'int64':
        tipo_pbi = "Número entero"
    elif tipo_python == 'float64':
        tipo_pbi = "Número decimal"
    elif tipo_python == 'object':
        tipo_pbi = "Texto"
    
    valores_unicos = df[col].nunique()
    nulos = df[col].isnull().sum()
    
    print(f"{col:20} | Python: {str(tipo_python):10} → PBI: {tipo_pbi}")
    print(f"{'':20} | Únicos: {valores_unicos}, Nulos: {nulos}")
    print()

---

## Solución Paso 2: Medidas DAX Completas

### Medida 1: Total de estudiantes

```dax
Total Estudiantes = 
    DISTINCTCOUNT(rendimiento_academico[id_estudiante])
```

**Explicación**: Usamos `DISTINCTCOUNT` porque un estudiante puede aparecer en múltiples filas (diferentes cursos/períodos). Si usáramos `COUNT`, contaríamos duplicados.

### Medida 2: Promedio de notas

```dax
Promedio Notas = 
    AVERAGE(rendimiento_academico[nota])
```

**Formato recomendado**: Número decimal, 2 decimales

### Medida 3: Tasa de aprobación

```dax
Tasa Aprobación = 
    VAR TotalRegistros = COUNT(rendimiento_academico[id_estudiante])
    VAR Aprobados = CALCULATE(
        COUNT(rendimiento_academico[id_estudiante]),
        rendimiento_academico[estado] = "Aprobado"
    )
    RETURN
        DIVIDE(Aprobados, TotalRegistros, 0)
```

**Formato**: Porcentaje, 1 decimal

**Nota**: Usamos variables (`VAR`) para hacer el código más legible.

### Medida 4: Cantidad de carreras

```dax
Carreras = DISTINCTCOUNT(rendimiento_academico[carrera])
```

### Medidas adicionales (avanzadas)

```dax
// Promedio del período anterior (para comparación)
Promedio Período Anterior = 
    CALCULATE(
        [Promedio Notas],
        PREVIOUSQUARTER(calendario[fecha])
    )

// Variación porcentual
Variación % = 
    VAR Actual = [Promedio Notas]
    VAR Anterior = [Promedio Período Anterior]
    RETURN
        DIVIDE(Actual - Anterior, Anterior, 0)
```

In [ ]:
# Verificar los cálculos esperados en Python
import pandas as pd

df = pd.read_csv('../datasets/universidad/rendimiento_academico.csv')

print("VALORES ESPERADOS PARA VERIFICAR EN POWER BI")
print("="*60)
print()

# Total estudiantes
total_est = df['id_estudiante'].nunique()
print(f"Total Estudiantes: {total_est:,}")

# Promedio notas
prom_notas = df['nota'].mean()
print(f"Promedio Notas: {prom_notas:.2f}")

# Tasa aprobación
if 'estado' in df.columns:
    total_registros = len(df)
    aprobados = (df['estado'] == 'Aprobado').sum()
    tasa = aprobados / total_registros * 100
    print(f"Tasa Aprobación: {tasa:.1f}%")

# Cantidad de carreras
n_carreras = df['carrera'].nunique()
print(f"Cantidad Carreras: {n_carreras}")

print()
print("-"*60)
print("Si sus valores en Power BI difieren, verifique los filtros aplicados.")

---

## Solución Paso 3: Layout Óptimo

### Configuraciones recomendadas por elemento

#### Tarjetas KPI

| Configuración | Valor |
|--------------|-------|
| Tamaño fuente valor | 32-40 pt |
| Tamaño fuente etiqueta | 10-12 pt |
| Mostrar etiqueta | Sí |
| Color de fondo | Blanco o gris muy claro |
| Borde | 1px gris claro |

#### Gráfico de barras horizontales

| Configuración | Valor |
|--------------|-------|
| Orientación | Horizontal (mejor para nombres largos) |
| Ordenamiento | Descendente por valor |
| Eje Y | Desde 0 siempre |
| Etiquetas de datos | Activadas, al final de la barra |
| Color | Un solo color, o degradado por valor |

#### Gráfico de líneas

| Configuración | Valor |
|--------------|-------|
| Marcadores | Activados si hay pocos puntos (<10) |
| Línea de referencia | Activada para mostrar meta |
| Eje X | Fecha o período |
| Grosor línea | 2-3 px |

In [ ]:
# Dashboard completo de ejemplo en Python (para comparar con Power BI)
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

df = pd.read_csv('../datasets/universidad/rendimiento_academico.csv')

# Configurar el estilo
plt.style.use('seaborn-v0_8-whitegrid')
fig = plt.figure(figsize=(16, 12))

# Título principal
fig.suptitle('Dashboard de Rendimiento Académico — FACES-UCAB', 
             fontsize=18, fontweight='bold', y=0.98)
fig.text(0.5, 0.95, 'Período: 2023-2024 | Actualizado: Enero 2025', 
         ha='center', fontsize=10, color='gray')

# Calcular métricas
total_est = df['id_estudiante'].nunique()
prom_notas = df['nota'].mean()
tasa_aprob = (df['estado'] == 'Aprobado').mean() * 100 if 'estado' in df.columns else 0
n_carreras = df['carrera'].nunique()

# KPIs en la parte superior
ax_kpi = fig.add_axes([0.05, 0.82, 0.9, 0.1])
ax_kpi.axis('off')

kpis = [
    ('Total Estudiantes', f'{total_est:,}', '#2563eb'),
    ('Promedio Notas', f'{prom_notas:.1f}', '#059669'),
    ('Tasa Aprobación', f'{tasa_aprob:.1f}%', '#7c3aed'),
    ('Carreras', f'{n_carreras}', '#ea580c')
]

for i, (label, value, color) in enumerate(kpis):
    x_pos = 0.125 + i * 0.25
    ax_kpi.add_patch(mpatches.FancyBboxPatch(
        (x_pos - 0.1, 0.1), 0.2, 0.8,
        boxstyle="round,pad=0.02,rounding_size=0.02",
        facecolor='white', edgecolor=color, linewidth=2
    ))
    ax_kpi.text(x_pos, 0.6, value, ha='center', va='center', 
                fontsize=24, fontweight='bold', color=color)
    ax_kpi.text(x_pos, 0.25, label, ha='center', va='center', 
                fontsize=10, color='gray')

# Gráfico de barras - Estudiantes por carrera
ax1 = fig.add_subplot(2, 2, 3)
estudiantes_carrera = df.groupby('carrera')['id_estudiante'].nunique().sort_values()
colors = ['#94a3b8'] * len(estudiantes_carrera)
colors[-1] = '#2563eb'  # Resaltar la mayor

bars = ax1.barh(estudiantes_carrera.index, estudiantes_carrera.values, color=colors)
ax1.set_xlabel('Cantidad de estudiantes')
ax1.set_title('Distribución de estudiantes por carrera', fontweight='bold', loc='left')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Etiquetas de datos
for bar, val in zip(bars, estudiantes_carrera.values):
    ax1.text(val + 5, bar.get_y() + bar.get_height()/2, 
             str(val), va='center', fontweight='bold')

# Gráfico de líneas - Tendencia por período
ax2 = fig.add_subplot(2, 2, 4)
if 'periodo' in df.columns:
    tendencia = df.groupby('periodo')['nota'].mean()
    ax2.plot(tendencia.index, tendencia.values, marker='o', 
             linewidth=2, color='#2563eb', markersize=8)
    ax2.axhline(y=14, color='#dc2626', linestyle='--', alpha=0.7, label='Meta: 14')
    ax2.set_ylabel('Promedio de notas')
    ax2.set_title('Tendencia del promedio por período', fontweight='bold', loc='left')
    ax2.legend()
    ax2.tick_params(axis='x', rotation=45)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Tabla resumen
ax3 = fig.add_subplot(2, 2, 1)
ax3.axis('off')
resumen = df.groupby('carrera').agg({
    'id_estudiante': 'nunique',
    'nota': 'mean'
}).round(1)
resumen.columns = ['Estudiantes', 'Promedio']
resumen = resumen.sort_values('Estudiantes', ascending=False).head(5)

table_data = [[idx] + list(row) for idx, row in resumen.iterrows()]
table = ax3.table(cellText=table_data,
                  colLabels=['Carrera', 'Estudiantes', 'Promedio'],
                  loc='center',
                  cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)
ax3.set_title('Top 5 Carreras por Matrícula', fontweight='bold', y=0.85)

plt.tight_layout(rect=[0, 0, 1, 0.92])
plt.show()

---

## Solución Paso 4: Configuración de Interactividad

### Matriz de interacciones recomendada

| Origen \ Destino | Tarjetas | Barras | Líneas | Tabla |
|-----------------|----------|--------|--------|-------|
| **Segmentador Carrera** | Filtrar | Filtrar | Filtrar | Filtrar |
| **Segmentador Período** | Filtrar | Filtrar | Filtrar | Filtrar |
| **Barras (clic)** | Filtrar | - | Filtrar | Filtrar |
| **Líneas (clic)** | Filtrar | Resaltar | - | Filtrar |

### Pasos para configurar en Power BI

1. Seleccionar el gráfico de origen (ej: Barras)
2. Ir a **Formato → Editar interacciones**
3. En cada gráfico destino, seleccionar el ícono:
   - 🔽 Embudo = Filtrar
   - 📊 Resaltar = Destacar sin ocultar
   - 🚫 Ninguno = Sin interacción
4. Repetir para cada gráfico como origen

### Configuración de Drill-Down

Para el gráfico de barras:
1. Arrastrar `carrera` al campo **Eje**
2. Arrastrar `semestre` como segundo nivel
3. En el encabezado del gráfico aparecen controles:
   - ↓ Bajar un nivel
   - ↑ Subir un nivel
   - ⇊ Expandir todo

### Tooltips personalizados

Para el gráfico de barras, agregar al tooltip:
- Promedio de notas de esa carrera
- Tasa de aprobación de esa carrera
- Semestre con mejor rendimiento

---

## Errores Comunes y Soluciones

### Error 1: Medida DAX da error de sintaxis

**Causa común**: Nombre de columna incorrecto

```dax
// INCORRECTO - si la columna se llama "nota_final"
Promedio = AVERAGE(rendimiento_academico[nota])

// CORRECTO
Promedio = AVERAGE(rendimiento_academico[nota_final])
```

**Solución**: Verificar nombre exacto en el panel de campos.

### Error 2: Tarjeta muestra "Error"

**Causa común**: División por cero en la medida

```dax
// INCORRECTO
Tasa = Aprobados / Total

// CORRECTO - usar DIVIDE que maneja el cero
Tasa = DIVIDE(Aprobados, Total, 0)
```

### Error 3: Segmentador no filtra correctamente

**Causa común**: Relación entre tablas no definida

**Solución**: 
1. Ir a Vista de Modelo
2. Verificar que hay una línea conectando las tablas
3. Si no existe, arrastrar el campo de una tabla a la otra para crear la relación

### Error 4: Gráfico de líneas no muestra tendencia temporal

**Causa común**: El eje X no se reconoce como fecha

**Solución**:
1. En Power Query, cambiar tipo de columna a Fecha
2. O crear una tabla calendario y relacionarla

### Error 5: Colores inconsistentes entre gráficos

**Solución**:
1. Ir a **Vista → Temas**
2. Seleccionar un tema predefinido
3. O **Personalizar tema actual** → definir paleta de colores

---

## Extensiones Avanzadas (Opcionales)

### 1. Agregar íconos condiciones a tarjetas

```dax
Indicador Tendencia = 
    VAR Actual = [Promedio Notas]
    VAR Anterior = [Promedio Período Anterior]
    RETURN
        IF(Actual > Anterior, "▲", IF(Actual < Anterior, "▼", "●"))
```

### 2. Crear página de detalle con drill-through

1. Crear nueva página "Detalle Carrera"
2. Agregar campo `carrera` como filtro de drill-through
3. En la página principal, clic derecho en una barra → Drill-through → Detalle Carrera

### 3. Agregar bookmarks para diferentes vistas

1. Configurar filtros para "Vista Ejecutivo" (solo KPIs)
2. Vista → Marcadores → Agregar
3. Repetir para "Vista Detallada" (todos los gráficos)
4. Agregar botones que cambien entre vistas

### 4. Formato condicional en tabla

1. Seleccionar la tabla
2. Formato → Color de fondo → Reglas
3. Configurar gradiente: rojo (bajo) → amarillo → verde (alto)

---

## Checklist de Verificación Final

Antes de entregar, verifique cada elemento:

### Datos
- [ ] Tipos de datos correctos en todas las columnas
- [ ] Sin errores de importación en Power Query
- [ ] Relaciones entre tablas definidas (si aplica)

### Medidas
- [ ] Total Estudiantes usa DISTINCTCOUNT
- [ ] Promedio Notas formateado con 2 decimales
- [ ] Tasa Aprobación formateada como porcentaje
- [ ] Todas las medidas tienen ícono de calculadora (fx)

### Visualizaciones
- [ ] 3 tarjetas KPI con etiquetas claras
- [ ] Gráfico de barras ordenado descendentemente
- [ ] Gráfico de líneas con eje X temporal
- [ ] Colores consistentes (paleta única)

### Interactividad
- [ ] Al menos 1 segmentador funcional
- [ ] Cross-filtering entre gráficos funciona
- [ ] Tooltips muestran información útil

### Presentación
- [ ] Título descriptivo (no genérico)
- [ ] Fecha de actualización visible
- [ ] Sin chartjunk (3D, sombras innecesarias, fondos de color)
- [ ] Legible en pantalla de presentación

---

## Recursos Adicionales

### Documentación oficial
- [Referencia de funciones DAX](https://docs.microsoft.com/dax/)
- [Guía de Power Query](https://docs.microsoft.com/power-query/)
- [Mejores prácticas de diseño](https://docs.microsoft.com/power-bi/visuals/power-bi-visualization-best-practices)

### Comunidad
- [Power BI Community](https://community.powerbi.com/)
- [DAX Patterns](https://www.daxpatterns.com/)

### Videos tutoriales
- Guy in a Cube (YouTube)
- Curbal (YouTube)
- SQLBI (YouTube)